<a href="https://colab.research.google.com/github/notmakgg-ux/GPT-build-by-me/blob/main/TrainingMyGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Building my own GPT from scratch.
 * Today I am learning and building my own GPT from scratch. This will help me
learn the base needed for LLM.
* The flow of the GPT is as follows :-

1.   Tokenization
2.   Embedding
3. Positional encoding
4. Attention
5. Transformer Block
6. GPT model
7. Data
7. Training
8. Evaluation



## 1. Tokenization

In [29]:
from dataclasses import dataclass
!pip install -q tiktoken
import tiktoken

In [30]:
@dataclass
class TokenizerConfig:
    name: str = "gpt2"
    vocab_size: int = 50257

In [31]:
class SimpleTokenizer:
  def __init__(self, config=None):
    self.config=config or TokenizerConfig()
    self.enc= tiktoken.get_encoding(self.config.name)
    self.eos_token="<|endoftext|>"
    self.eos_token_id=self.enc.encode(
        self.eos_token,  allowed_special={self.eos_token}
    )[0]
  def encode(self, text):
    return self.enc.encode(text , allowed_special={self.eos_token})

  def decode(self,ids):
    return self.enc.decode(ids)

  @property
  def vocab_size(self):
    return self.enc.n_vocab

In [32]:
tokenizer = SimpleTokenizer()
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"EOS token: {tokenizer.eos_token}")
print(f"EOS token ID: {tokenizer.eos_token_id}")
print()

text = "The quick brown fox jumps over the lazy dog."
print(f"Original text: {text}")
encoded = tokenizer.encode(text)
print(f"Encoded text: {encoded}")
decoded = tokenizer.decode(encoded)
print(f"Decoded text: {decoded}")
print(f"Roundtrip: {text == decoded}")


Vocab size: 50257
EOS token: <|endoftext|>
EOS token ID: 50256

Original text: The quick brown fox jumps over the lazy dog.
Encoded text: [464, 2068, 7586, 21831, 18045, 625, 262, 16931, 3290, 13]
Decoded text: The quick brown fox jumps over the lazy dog.
Roundtrip: True


In [33]:
rare = tokenizer.encode("antidisestablishmentarianism")
pieces = [tokenizer.decode([t]) for t in rare]
print(f"Pieces: {pieces}")
print(f"Decoded: '{tokenizer.decode(rare)}'")
print()

text = tokenizer.encode("Hello world!")
print(f"Decoded: {tokenizer.decode(text)}")
print()
emoji = tokenizer.encode("Hello \U0001f60a world")
print(f"Emoji test: {tokenizer.decode(emoji)}")

Pieces: ['ant', 'idis', 'establishment', 'arian', 'ism']
Decoded: 'antidisestablishmentarianism'

Decoded: Hello world!

Emoji test: Hello 😊 world


## 2. Embedding

In [34]:
import torch
import torch.nn as nn
import math

In [35]:
class Embedding(nn.Module):
  def __init__(self, vocab_size, d_model):
    super().__init__()
    self.embed=nn.Embedding(vocab_size,d_model)

  def forward(self,x):
    return self.embed(x)


In [36]:
vocab_size = 1000
d_model = 768

embedding = Embedding(vocab_size, d_model)
print(f"Embedding table shape: [{vocab_size}, {d_model}]")
print(f"Total parameters: {sum(p.numel() for p in embedding.parameters()):,}")
print()

token_ids = torch.tensor([[123, 45, 678, 464, 890]])
print(f"Input token IDs: {token_ids}")
print(f"Input shape: {token_ids.shape}")
print()

output = embedding(token_ids)
print(f"Output shape: {output.shape}")
print(f"Each token is now a {d_model}-dimensional vector")
print(f"First token vector preview: {output[0, 0, :5].tolist()} ...")

Embedding table shape: [1000, 768]
Total parameters: 768,000

Input token IDs: tensor([[123,  45, 678, 464, 890]])
Input shape: torch.Size([1, 5])

Output shape: torch.Size([1, 5, 768])
Each token is now a 768-dimensional vector
First token vector preview: [0.14034005999565125, -0.5157710313796997, 0.4746512770652771, -0.6638162732124329, -0.7720792889595032] ...


## 3. Postional Encoding Using RoPE

In [37]:
import torch
import torch.nn as nn
import math

In [38]:
class RotaryPositionalEmbedding(nn.Module):
  def __init__(self, d_model, max_seq_len=2048, theta=10000.0):
    super().__init__()
    assert d_model%2==0, "d_model must be even"


    dim_indices=torch.arange(0, d_model, 2).float()
    inv_freq=1.0 / (theta ** (dim_indices / d_model))

    positions = torch.arange(max_seq_len).float()
    freqs=torch.outer(positions, inv_freq)
    emb = freqs.repeat_interleave(2, dim=-1)

    self.register_buffer("cos_cached", emb.cos())
    self.register_buffer("sin_cached", emb.sin())


  @staticmethod
  def rotate_half(x):

    x_even = x[..., 0::2]
    x_odd  = x[..., 1::2]
    return torch.stack([-x_odd, x_even], dim=-1).flatten(-2)

  def forward(self, x, seq_len):

    cos = self.cos_cached[:seq_len].to(x.dtype).unsqueeze(0).unsqueeze(0)
    sin = self.sin_cached[:seq_len].to(x.dtype).unsqueeze(0).unsqueeze(0)
    return (x * cos) + (self.rotate_half(x) * sin)





In [39]:
rope = RotaryPositionalEmbedding(d_model=4, max_seq_len=16)

q = torch.tensor([[[[0.8, 0.3, -0.5, 0.2]]]])  # [1, 1, 1, 4]
print(f"Original vector: {q[0, 0, 0].tolist()}")
print()

for pos in [0, 1, 2, 5]:
    rotated = rope(q, seq_len=pos + 1)
    last = rotated[0, 0, -1].tolist()
    vals = [f"{v:.3f}" for v in last]
    print(f"Position {pos}: [{', '.join(vals)}]")

Original vector: [0.800000011920929, 0.30000001192092896, -0.5, 0.20000000298023224]

Position 0: [0.800, 0.300, -0.500, 0.200]
Position 1: [0.180, 0.835, -0.502, 0.195]
Position 2: [-0.606, 0.603, -0.504, 0.190]
Position 5: [0.515, -0.682, -0.509, 0.175]


In [40]:

rope = RotaryPositionalEmbedding(d_model=64, max_seq_len=256)

seq_len = 12
# SAME vector at every position, so only the rotation differs
q0 = torch.randn(1, 1, 1, 64).expand(1, 1, seq_len, 64)
k0 = torch.randn(1, 1, 1, 64).expand(1, 1, seq_len, 64)

q_rot = rope(q0, seq_len)
k_rot = rope(k0, seq_len)

def score(i, j):
    return (q_rot[0, 0, i] @ k_rot[0, 0, j]).item()

print(f"Distance 2, positions (2,4): {score(2, 4):.4f}")
print(f"Distance 2, positions (5,7): {score(5, 7):.4f}")
print(f"Distance 2, positions (0,2): {score(0, 2):.4f}")
print(f"Distance 8, positions (0,8): {score(0, 8):.4f}")
print()
print("Same distance -> same score. RoPE encodes RELATIVE position.")
assert math.isclose(score(2, 4), score(5, 7), rel_tol=1e-4, abs_tol=1e-3)
assert math.isclose(score(2, 4), score(0, 2), rel_tol=1e-4, abs_tol=1e-3)

Distance 2, positions (2,4): 17.7848
Distance 2, positions (5,7): 17.7848
Distance 2, positions (0,2): 17.7848
Distance 8, positions (0,8): 5.3835

Same distance -> same score. RoPE encodes RELATIVE position.


## 4. Attention

In [41]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [42]:
def create_causal_mask(seq_len, device):
    mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
    return mask.view(1, 1, seq_len, seq_len)

In [43]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1,
                 use_rope=True, max_seq_len=2048):
        super().__init__()

        assert d_model % num_heads == 0, \
            "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.use_rope = use_rope

        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)


        self.rope = (
            RotaryPositionalEmbedding(self.head_dim, max_seq_len)
            if use_rope else None
        )

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)


        self.store_attn = False
        self.last_attn = None

    def forward(self, x, mask=None):
        batch_size, seq_len, d_model = x.shape

        qkv = self.qkv_proj(x)
        qkv = qkv.reshape(batch_size, seq_len, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        if self.use_rope:
            q = self.rope(q, seq_len)
            k = self.rope(k, seq_len)

        attn_weights = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        if mask is not None:
            attn_weights = attn_weights.masked_fill(mask == 0, float("-inf"))

        attn_weights = F.softmax(attn_weights, dim=-1)

        if self.store_attn:
            self.last_attn = attn_weights.detach().cpu()

        attn_weights = self.attn_dropout(attn_weights)
        attn_output = attn_weights @ v

        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.reshape(batch_size, seq_len, self.d_model)

        output = self.out_proj(attn_output)
        output = self.resid_dropout(output)
        return output

In [44]:
attn = MultiHeadAttention(d_model=768, num_heads=12)

batch_size = 8
seq_len = 64
x = torch.randn(batch_size, seq_len, 768)
mask = create_causal_mask(seq_len, x.device)

output = attn(x, mask)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Input and output shapes match: {x.shape == output.shape}")
print()

params = sum(p.numel() for p in attn.parameters())
print(f"Attention parameters: {params:,}")


Input shape:  torch.Size([8, 64, 768])
Output shape: torch.Size([8, 64, 768])
Input and output shapes match: True

Attention parameters: 2,359,296


In [45]:
tiny_attn = MultiHeadAttention(d_model=4, num_heads=1)

tokens = torch.tensor([[
    [0.5,  0.2, -0.3,  0.8],
    [0.1, -0.5,  0.7, -0.2],
    [0.9,  0.3, -0.1, -0.5],
]])
print(f"Input tokens shape: {tokens.shape}")
print()

with torch.no_grad():
    qkv = tiny_attn.qkv_proj(tokens)
    qkv = qkv.reshape(1, 3, 3, 1, 4)
    qkv = qkv.permute(2, 0, 3, 1, 4)
    q, k, v = qkv[0], qkv[1], qkv[2]

print("Query vectors:")
for i in range(3):
    vals = [f"{v:.3f}" for v in q[0, 0, i].tolist()]
    print(f"  Token {i}: [{', '.join(vals)}]")
print()

print("Key vectors:")
for i in range(3):
    vals = [f"{v:.3f}" for v in k[0, 0, i].tolist()]
    print(f"  Token {i}: [{', '.join(vals)}]")
print()

scores = (q @ k.transpose(-2, -1)) / math.sqrt(4)
print("Attention scores (before mask and softmax):")
print(f"  Token 0: {[f'{v:.3f}' for v in scores[0, 0, 0].tolist()]}")
print(f"  Token 1: {[f'{v:.3f}' for v in scores[0, 0, 1].tolist()]}")
print(f"  Token 2: {[f'{v:.3f}' for v in scores[0, 0, 2].tolist()]}")

Input tokens shape: torch.Size([1, 3, 4])

Query vectors:
  Token 0: [0.211, -0.306, 0.270, -0.144]
  Token 1: [0.080, 0.159, 0.034, 0.406]
  Token 2: [-0.446, 0.340, 0.222, -0.531]

Key vectors:
  Token 0: [-0.095, -0.120, -0.113, -0.385]
  Token 1: [0.003, 0.095, -0.177, 0.096]
  Token 2: [0.358, -0.153, -0.091, 0.065]

Attention scores (before mask and softmax):
  Token 0: ['0.021', '-0.045', '0.044']
  Token 1: ['-0.093', '0.024', '0.014']
  Token 2: ['0.091', '-0.030', '-0.133']


In [46]:
mask = create_causal_mask(3, scores.device)
scores_masked = scores.masked_fill(mask == 0, float('-inf'))
weights = F.softmax(scores_masked, dim=-1)

print("Attention weights (after mask and softmax):")
for i in range(3):
    row = [f"{v:.3f}" for v in weights[0, 0, i].tolist()]
    print(f"  Token {i} attends to: [{', '.join(row)}]")
print()
print("Token 0 sees only itself. Token 1 sees 0 and 1. Token 2 sees all three.")
print()

output = weights @ v
print("Output vectors (weighted sum of values):")
for i in range(3):
    vals = [f"{v:.3f}" for v in output[0, 0, i].tolist()]
    print(f"  Token {i}: [{', '.join(vals)}]")

Attention weights (after mask and softmax):
  Token 0 attends to: [1.000, 0.000, 0.000]
  Token 1 attends to: [0.471, 0.529, 0.000]
  Token 2 attends to: [0.372, 0.330, 0.298]

Token 0 sees only itself. Token 1 sees 0 and 1. Token 2 sees all three.

Output vectors (weighted sum of values):
  Token 0: [0.665, -0.053, 0.562, -0.392]
  Token 1: [0.154, -0.013, 0.199, -0.246]
  Token 2: [0.247, 0.128, 0.166, -0.236]


In [47]:
print("Causal attention matrix (Token i attending to Token j):")
print("          T0      T1      T2")
for i in range(3):
    row = "  ".join([f"{weights[0, 0, i, j]:.3f}" for j in range(3)])
    print(f"T{i}       {row}")
print()
print("The upper right is zero. The model cannot see the future.")

Causal attention matrix (Token i attending to Token j):
          T0      T1      T2
T0       1.000  0.000  0.000
T1       0.471  0.529  0.000
T2       0.372  0.330  0.298

The upper right is zero. The model cannot see the future.


## 5. Transformer             

In [48]:
class RMSNorm(nn.Module):
  def __init__(self,d_model,eps=1e-6):
    super().__init__()
    self.weight=nn.Parameter(torch.ones(d_model))
    self.eps=eps

  def forward(self,x):
    rms= torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
    return x * self.weight * rms

In [49]:
class SwiGLU(nn.Module):
  def __init__(self, d_model, hidden_dim=None):
    super().__init__()
    if hidden_dim is None:
      hidden_dim = int(8 * d_model / 3)
      hidden_dim = 64 * ((hidden_dim + 63) // 64)   # round up to a multiple of 64
    self.w1 = nn.Linear(d_model, hidden_dim, bias=False)
    self.w2 = nn.Linear(d_model, hidden_dim, bias=False)
    self.w3 = nn.Linear(hidden_dim, d_model, bias=False)

  def forward(self, x):
    return self.w3(F.silu(self.w1(x)) * self.w2(x))

In [50]:
def make_norm(norm_type, d_model):
    if norm_type == "rmsnorm":
        return RMSNorm(d_model)
    elif norm_type == "layernorm":
        return nn.LayerNorm(d_model)
    raise ValueError(f"Unknown norm_type: {norm_type}")


class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1,
                 use_rope=True, norm_type="rmsnorm", max_seq_len=2048):
        super().__init__()
        self.norm1 = make_norm(norm_type, d_model)
        self.attention = MultiHeadAttention(
            d_model, num_heads, dropout, use_rope, max_seq_len
        )
        self.norm2 = make_norm(norm_type, d_model)
        self.ffn = SwiGLU(d_model)

    def forward(self, x, mask=None):
        x = x + self.attention(self.norm1(x), mask)
        x = x + self.ffn(self.norm2(x))
        return x

In [51]:
d_model = 768
num_heads = 12
seq_len = 64
batch_size = 4

block = TransformerBlock(d_model, num_heads)

x = torch.randn(batch_size, seq_len, d_model)
mask = create_causal_mask(seq_len, x.device)

output = block(x, mask)

print(f"Input shape:  {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Shapes match: {x.shape == output.shape}")
print()

diff = (output - x).abs().mean().item()
print(f"Mean change after block: {diff:.4f}")
print(f"The block changed the input (it did something)")
print()

params = sum(p.numel() for p in block.parameters())
print(f"Parameters per block: {params:,}")
print(f"12 blocks: {params * 12:,} params (excluding embeddings)")

Input shape:  torch.Size([4, 64, 768])
Output shape: torch.Size([4, 64, 768])
Shapes match: True

Mean change after block: 0.1128
The block changed the input (it did something)

Parameters per block: 7,079,424
12 blocks: 84,953,088 params (excluding embeddings)


In [52]:
block.eval()
with torch.no_grad():
    for p in block.attention.parameters():
        p.zero_()
    for p in block.ffn.parameters():
        p.zero_()

x = torch.randn(1, 4, d_model)
mask = create_causal_mask(4, x.device)
output = block(x, mask)

diff = (output - x).abs().max().item()
print(f"Max difference when weights are zero: {diff:.10f}")
print(f"Input passes through unchanged (the residual works)")

Max difference when weights are zero: 0.0000000000
Input passes through unchanged (the residual works)


## 6. GPT Model

In [53]:
@dataclass
class GPTConfig:
    vocab_size: int = 50257
    max_seq_len: int = 256
    d_model: int = 384
    num_heads: int = 6
    num_layers: int = 6
    dropout: float = 0.1
    pos_type: str = "rope"        # "rope" or "learned"
    norm_type: str = "rmsnorm"    # "rmsnorm" or "layernorm"

In [54]:
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        c = config
        assert c.pos_type in ("rope", "learned")
        assert c.norm_type in ("rmsnorm", "layernorm")

        self.tok_emb = Embedding(c.vocab_size, c.d_model)


        self.pos_emb = (
            nn.Embedding(c.max_seq_len, c.d_model)
            if c.pos_type == "learned" else None
        )
        self.drop = nn.Dropout(c.dropout)

        self.blocks = nn.ModuleList([
            TransformerBlock(
                c.d_model, c.num_heads, c.dropout,
                use_rope=(c.pos_type == "rope"),
                norm_type=c.norm_type,
                max_seq_len=c.max_seq_len,
            )
            for _ in range(c.num_layers)
        ])

        self.final_norm = make_norm(c.norm_type, c.d_model)
        self.lm_head = nn.Linear(c.d_model, c.vocab_size, bias=False)


        self.lm_head.weight = self.tok_emb.embed.weight


        self.register_buffer(
            "causal_mask",
            create_causal_mask(c.max_seq_len, device="cpu"),
            persistent=False,
        )

        self.apply(self._init_weights)

        for name, p in self.named_parameters():
            if name.endswith("out_proj.weight") or name.endswith("w3.weight"):
                nn.init.normal_(p, mean=0.0,
                                std=0.02 / math.sqrt(2 * c.num_layers))

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.config.max_seq_len, "sequence longer than max_seq_len"

        x = self.tok_emb(idx)
        if self.pos_emb is not None:
            x = x + self.pos_emb(torch.arange(T, device=idx.device))
        x = self.drop(x)

        mask = self.causal_mask[:, :, :T, :T]
        for block in self.blocks:
            x = block(x, mask)

        x = self.final_norm(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)), targets.view(-1)
            )
        return logits, loss

    def num_params(self):

        return sum(p.numel() for p in self.parameters())

In [55]:
torch.manual_seed(0)
print(f"Random-guess loss = ln(vocab) = {math.log(50257):.3f}\n")

for pos in ["rope", "learned"]:
    for norm in ["rmsnorm", "layernorm"]:
        cfg = GPTConfig(pos_type=pos, norm_type=norm)
        model = GPT(cfg).eval()

        idx = torch.randint(0, cfg.vocab_size, (2, 32))
        targets = torch.randint(0, cfg.vocab_size, (2, 32))

        with torch.no_grad():
            logits, loss = model(idx, targets)


            idx2 = idx.clone()
            idx2[:, -1] = (idx2[:, -1] + 1) % cfg.vocab_size
            logits2, _ = model(idx2)
            causal_ok = torch.allclose(logits[:, :-1], logits2[:, :-1], atol=1e-5)

        print(f"{pos:8s} + {norm:9s} | params: {model.num_params():,} | "
              f"logits: {tuple(logits.shape)} | init loss: {loss.item():.3f} | "
              f"causal: {causal_ok}")

Random-guess loss = ln(vocab) = 10.825

rope     + rmsnorm   | params: 29,920,512 | logits: (2, 32, 50257) | init loss: 10.877 | causal: True
rope     + layernorm | params: 29,925,504 | logits: (2, 32, 50257) | init loss: 10.896 | causal: True
learned  + rmsnorm   | params: 30,018,816 | logits: (2, 32, 50257) | init loss: 10.866 | causal: True
learned  + layernorm | params: 30,023,808 | logits: (2, 32, 50257) | init loss: 10.946 | causal: True


## 7. Data

In [56]:
import os, urllib.request
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("Warning: no GPU. In Colab use Runtime -> Change runtime type -> T4 GPU")

Using device: cuda


In [57]:
DATA_SOURCE = "tinystories"   # "tinystories", "shakespeare", or "custom"

URLS = {
    "tinystories": "https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-valid.txt",
    "shakespeare": "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
}

if DATA_SOURCE == "custom":
    # Upload your own .txt file in the Colab file panel and set the name here
    data_path = "my_text.txt"
else:
    data_path = f"{DATA_SOURCE}.txt"
    if not os.path.exists(data_path):
        print("Downloading...")
        urllib.request.urlretrieve(URLS[DATA_SOURCE], data_path)

with open(data_path, "r", encoding="utf-8") as f:
    text = f.read()

print(f"Characters: {len(text):,}")
print(f"Preview:\n{text[:400]}")

Downloading...
Characters: 22,493,387
Preview:
u don't have to be scared of the loud dog, I'll protect you". The mole felt so safe with the little girl. She was very kind and the mole soon came to trust her. He leaned against her and she kept him safe. The mole had found his best friend.
<|endoftext|>
Once upon a time, in a warm and sunny place, there was a big pit. A little boy named Tom liked to play near the pit. One day, Tom lost his red b


In [58]:
tokenizer = SimpleTokenizer()

ids = tokenizer.encode(text)
data = torch.tensor(ids, dtype=torch.long)
print(f"Total tokens: {len(data):,}")
print(f"Chars per token: {len(text) / len(data):.2f}")


split = int(0.9 * len(data))
train_data = data[:split]
val_data = data[split:]
print(f"Train tokens: {len(train_data):,} | Val tokens: {len(val_data):,}")

Total tokens: 5,532,654
Chars per token: 4.07
Train tokens: 4,979,388 | Val tokens: 553,266


In [59]:
def get_batch(split, batch_size, block_size, generator=None):
    src = train_data if split == "train" else val_data
    starts = torch.randint(0, len(src) - block_size - 1, (batch_size,),
                           generator=generator)
    x = torch.stack([src[s : s + block_size] for s in starts])
    y = torch.stack([src[s + 1 : s + block_size + 1] for s in starts])
    return x.to(device), y.to(device)

In [60]:
x, y = get_batch("train", batch_size=2, block_size=12)
print(f"x shape: {x.shape} | y shape: {y.shape}\n")

for b in range(2):
    print(f"Sample {b}")
    print("  input :", repr(tokenizer.decode(x[b].tolist())))
    print("  target:", repr(tokenizer.decode(y[b].tolist())))

print("\nCheck: target is the input shifted left by one token ->",
      torch.equal(x[0, 1:], y[0, :-1]))

x shape: torch.Size([2, 12]) | y shape: torch.Size([2, 12])

Sample 0
  input : ' a time, there was a green bird. The green bird'
  target: ' time, there was a green bird. The green bird wanted'
Sample 1
  input : ' always took good care of her.\nOne sunny day,'
  target: ' took good care of her.\nOne sunny day, B'

Check: target is the input shifted left by one token -> True
